<div align="center">
    <h1>k-Nearest Neighbors: Real-World Application and Hyperparameter Tuning</h1>
    <hr>
    <p><strong>Moving from toy datasets to real images!</strong></p>
    <p>In Part 1, we built intuition with 2D data. Now we'll apply k-NN to real photographs from CIFAR-10, a classic image-classification dataset. Along the way, we'll learn one of the most important skills in machine learning: <strong>hyperparameter tuning using cross-validation</strong>.</p>
    <p><em>Key Learning Objectives:</em> Understanding why hyperparameters matter • Implementing cross-validation • Selecting metrics for real-world requirements</p>
</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams['image.interpolation'] = 'nearest'

%load_ext autoreload
%autoreload 2

## The CIFAR-10 Dataset

CIFAR-10 is a classic image-classification dataset and one of the most widely used benchmarks in machine learning. It is small enough to download and experiment with quickly, so it is a common first dataset for image classifiers:

- **60,000 color images** of 32×32 pixels: 50,000 for training and 10,000 for testing
- **10 classes** of everyday objects
- **Balanced classes**: 6,000 images per class

<br>

<div align="center">

| Label | Class | Label | Class |
|:-----:|:-----:|:-----:|:-----:|
| 0 | ✈️ Airplane | 5 | 🐕 Dog |
| 1 | 🚗 Car | 6 | 🐸 Frog |
| 2 | 🐦 Bird | 7 | 🐴 Horse |
| 3 | 🐱 Cat | 8 | 🚢 Ship |
| 4 | 🦌 Deer | 9 | 🚚 Truck |

</div>

<br>

The first time you run the cell below, it downloads about 170 MB into `data/datasets/CIFAR10` and shows a progress bar. Later runs reuse the downloaded files.

## Data Preprocessing

k-NN compares images only through the Euclidean distance between their pixel values, so preprocessing decides what "similar" means. The pipeline has three steps.

#### 1. Subsampling
We load a balanced subset with **800 training, 100 validation, and 100 test images per class**, that is 8,000 / 1,000 / 1,000 images. k-NN computes the distance from every query to every stored training image, and cross-validation below repeats this many times, so a smaller subset keeps the experiments fast. The validation images come from the original training set; the test images come from the official test set.

#### 2. Normalization
We compute **one mean and one standard deviation per color channel** (red, green, blue) over all pixels of all training images, and use them to normalize the training, validation, and test images:

$$\tilde{x}_{c} = \frac{x_{c} - \mu_c}{\sigma_c}, \qquad c \in \{\text{R}, \text{G}, \text{B}\}.$$

As in Part 1, the statistics come from the training images only. This is the standard preprocessing for image classifiers. Convolutional networks trained on ImageNet, for example, use exactly this with the ImageNet channel means and standard deviations, and we will use it in later labs too.

Will it help k-NN? Much less than in Part 1. There, the features had different units, and normalization fixed that. Here every feature is a pixel intensity between 0 and 255 with a similar spread. Subtracting the same means from every image does not change any distance, and dividing all channels by similar standard deviations is almost a uniform rescaling, so the nearest neighbors barely change. Normalization matters much more for models trained with gradient descent, which you will meet in the next labs.

#### 3. Flattening
Each normalized 32×32×3 image becomes a vector of **3,072 numbers**, one per pixel and color channel. k-NN treats each of these numbers as a separate feature.

### Lessons from Part 1
Keep the failure modes from Part 1 in mind when you look at the results:

- **Many features, few points (Section 3).** Each image has 3,072 features, but we store only 8,000 training images. In Part 1, with 500 random features, even the nearest of 1M points was almost as far away as the mean distance. Real images are not random, because neighboring pixels are strongly correlated, so k-NN does better than guessing. Still, expect accuracy far below the 98% from the 2D dataset.
- **Distance is not the same as similarity (Sections 1 and 2).** Pixel distance only checks whether the same pixel positions have similar values. The same cat shifted by a few pixels or photographed in darker light can be far away, while a bird and an airplane on the same blue sky can be close. Like the noise in Part 1, this puts images with other labels among the nearest neighbors.

Let's load and prepare our data:

In [ ]:
import os

from utils import load_cifar10_subset, reshape_to_vectors, normalize_per_channel, dataset_stats

notebook_dir = os.getcwd()
dataset_path = os.path.join(notebook_dir, 'data', 'datasets', 'CIFAR10')

X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10_subset(directory=dataset_path,
                                                                     num_train=800, 
                                                                     num_val=100, 
                                                                     num_test=100,
                                                                     visualize_samples=True)

# Print out the dataset statistics and visualize a few samples
num_features, num_classes, num_samples = dataset_stats(X_train, y_train, X_val, y_val, X_test, y_test, verbose=True)

# Normalize each color channel with training statistics, then reshape the images to vectors
X_train, X_val, X_test = normalize_per_channel(X_train, X_val, X_test)
X_train, X_val, X_test = reshape_to_vectors(X_train, X_val, X_test)

## Baseline Performance: k-NN on Real Images

### Setting Expectations
Now we'll apply our k-NN classifier to real image data. Notice the dramatic difference from our 2D toy dataset:
- **Much lower accuracy** (~30% vs 98%) - why?
- **High-dimensional curse**: 3072 dimensions vs 2 dimensions
- **Complex patterns**: Real images have texture, lighting, occlusion

### Why k=5?
We arbitrarily chose k=5 as a starting point. But is this optimal? Probably not! This motivates our next section on hyperparameter tuning.

Let's see how our classifier performs with this initial guess:

In [ ]:
from sklearn.metrics import accuracy_score
from assignments import KNNClassifier

# Create and train the classifier
knn = KNNClassifier(k=5, vectorized=True)
knn.train(X_train, y_train)

# Predict the labels of the given samples
y_pred = knn.predict(X_test)

# Compute the accuracy of the predictions, you should expect to see an accuracy of around 0.29.
print(f'Accuracy: {accuracy_score(y_test, y_pred):.2f}')

### Where Does k-NN Make Mistakes?

Accuracy is a single number: it tells us how often k-NN is wrong, but not **which classes it confuses**. A **confusion matrix** shows this. Row $i$, column $j$ counts the test images of class $i$ that k-NN predicted as class $j$:

- The **diagonal** holds the correct predictions.
- Everything **off the diagonal** is a mistake. A large off-diagonal number means two classes are often confused.
- Each **row** sums to the number of test images of that class (100 here). Each **column** sums to how often k-NN predicted that class.

The color shows each count as a share of its row, so a dark diagonal cell means the class is recognized well.


In [ ]:
from utils import plot_confusion_matrix

classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# Rows are the true classes, columns are the predicted classes
plot_confusion_matrix(y_test, y_pred, classes)


**Think about it:** Which classes does k-NN recognize best, and which worst? Which pairs of classes are confused most often, and what might their images have in common in terms of colors or backgrounds? Is any class predicted much more often than it really appears? Remember that k-NN only compares pixel values.


## Hyperparameter Tuning: Finding the Best k

### The Central Question
How do we choose k? Too small and we're sensitive to noise. Too large and we lose local patterns. The answer: **let the data tell us through cross-validation!**

### Understanding Cross-Validation
Cross-validation is a systematic way to evaluate different hyperparameters:

1. **Split training data into folds** (e.g., 5 equal parts)
2. **For each k value**:
   - Train on 4 folds, validate on 1 fold
   - Repeat 5 times (each fold gets to be validation once)
   - Average the results
3. **Choose k with best average performance**

### Why Cross-Validation Works
- **Uses all data**: Every sample is used for both training and validation
- **Reduces overfitting**: Performance estimated on unseen data
- **Stable estimates**: Averaging reduces variance from random splits

### Implementation Task
Before proceeding, implement the `cross_validate_knn` function in `assignments/tuning.py`. This function should:
1. Split data into folds
2. Evaluate each k value using the fold rotation described above
3. Compute metrics (accuracy, precision, recall) for each configuration

Let's run cross-validation to find the optimal k:

In [ ]:
from assignments import cross_validate_knn

num_folds = 5
k_choices = np.array([1, 3, 5, 8, 10, 12, 15, 20, 30], dtype=np.int32)

# Find the best value of k using cross validation
k_to_metrics = cross_validate_knn(classifier=knn, X=X_train, y=y_train, k_choices=k_choices, num_folds=num_folds)

## Choosing the Right Metric: Context Matters!

### The Key Insight
There's **no universally "best" hyperparameter or metric**. What's optimal depends entirely on your specific application and requirements. This is a fundamental principle in machine learning that extends beyond k-NN to all algorithms.

### Understanding Classification Metrics

#### **Accuracy**
- **Formula**: (True Positives + True Negatives) / Total
- **Use when**: Classes are balanced and all errors are equally costly
- **Limitation**: Can be misleading with imbalanced data

#### **Precision** 
- **Formula**: True Positives / (True Positives + False Positives)
- **Interpretation**: "When we predict positive, how often are we right?"
- **In the confusion matrix**: the diagonal entry divided by its **column** sum
- **Use when**: False positives are costly (e.g., spam detection)

#### **Recall** (Sensitivity)
- **Formula**: True Positives / (True Positives + False Negatives)  
- **Interpretation**: "Of all actual positives, how many did we find?"
- **In the confusion matrix**: the diagonal entry divided by its **row** sum
- **Use when**: Missing positives is costly (e.g., disease detection)

#### **F1 Score**
- **Formula**: 2 × (Precision × Recall) / (Precision + Recall)
- **Interpretation**: Harmonic mean of precision and recall
- **Use when**: You need balance between precision and recall

### Analyzing Our Results
The plots below show how different k values affect various metrics. Notice:
- Different metrics may suggest different optimal k values
- Performance varies across classes
- Trade-offs exist (improving one metric might worsen another)

Let's visualize our cross-validation results:

In [ ]:
from utils import plot_knn_cross_validation

# Plot the cross validation results
plot_knn_cross_validation(k_to_metrics, classes)

## Real-World Scenario: Ship Detection System

### The Business Context
You're developing an image classification system for a maritime security company. They monitor ports and need to automatically detect ships in surveillance images. Let's understand their requirements through a conversation:

### Requirements Gathering

**You:** "What metrics should we use to evaluate the classifier's performance?"

**Company:** "We're not familiar with technical metrics. Can you explain?"

**You:** "Metrics help us measure how well the classifier works. For instance, accuracy tells us the percentage of correct predictions overall."

**Company:** "Perfect! We want maximum accuracy."

**You:** "I'll optimize for accuracy. Any other requirements?"

**Company:** "Actually, yes - ship detection is critical for us. We absolutely cannot miss any ships in our surveillance."

**You:** "What if the system incorrectly flags something as a ship when it isn't?"

**Company:** "That's acceptable - we can manually verify. But missing an actual ship could be a security breach."

### Translating Business Needs to ML Metrics

From this conversation, we learn:
1. **Primary concern**: Never miss actual ships (minimize false negatives)
2. **Secondary concern**: False alarms are tolerable (false positives acceptable)


<div align="center">
      <hr>
</div>

## Assignment Questions

Based on the scenario above and your cross-validation results, answer these questions:

### Question 1: Metric Selection
**Which metric is most suitable for this ship detection task? Explain your reasoning.**

*Hint: Consider what the company values most - catching all ships vs. overall accuracy.*

### Question 2: Optimal k Value
**Based on your analysis, what value of k would you recommend for this task?**

*Hint: Look at the cross-validation plots and find which k maximizes your chosen metric for the relevant class.*

### Your Answers:

*(Write your answers here after analyzing the results)*

1. **Chosen Metric**: ...
   - **Reasoning**: ...
2. **Recommended k**: ...
   - **Evidence from plots**: ...

<div align="center">
      <hr>
</div>

## Key Takeaways

- **No universal best**: The "optimal" hyperparameter depends on your specific task
- **Metrics matter**: Different metrics optimize for different aspects of performance
- **Business requirements drive technical choices**: Always align your ML decisions with real-world needs
- **Cross-validation is essential**: It provides robust estimates and helps avoid overfitting